In [ ]:
#importacion de librerias necesarias 
import pandas as pd
import numpy as np
import json
#Sensores usados 
#Humedad del ambiente DHT22
#temperatura del ambiente 
#humedad del suelo 
#porcentaje de lluvia 


In [ ]:
#Fase 1: Limpieza de datos 

#Fase 1.1: Configuracion inicial 
config = {
    "frecuencia_esperada": "10s",

    "rangos_validos": {
        "temp_ambiente": (-10, 60),   # °C
        "hum_ambiente": (0, 100),     
        "hum_suelo": (0, 100),        
        "lluvia_pct": (0, 100),       
    },

    "error_values": [-999, 999, -1, 9999, -9999],
    "columnas_cero": ["hum_ambiente", "hum_suelo"],


    "max_delta_por_lectura": {
        "temp_ambiente": 1.5,   
        "hum_ambiente": 5,      
        "hum_suelo": 2,         
    },

    "ventana_promedio_relleno": "5min",
    "limite_relleno": 30,  #30 lecturas 
    "limite_relleno_lluvia": 2, #repite el dato por dos lecturas consecutivas (20s) si dura mas NaN
    "columnas_numericas": ["temp_ambiente", "hum_ambiente", "hum_suelo", "lluvia_pct"],
}

# Fase 1.2: Carga segura de JSON 
def cargar_json_seguro(ruta_o_lista): 
    registros_validos = []
    registros_descartados = []

    if isinstance(ruta_o_lista, str): 
        with open(ruta_o_lista, "r", encoding="utf-8") as f:
            contenido = f.read().strip()
        try:
            data = json.loads(contenido)
            crudos = data if isinstance(data, list) else [data]    #coma rota comillas malas la guarda en registros_descartados para revisarlo despues
        except json.JSONDecodeError:
            crudos = []
            for i, linea in enumerate(contenido.splitlines()): #lee archivo renglon a renglon 
                linea = linea.strip().rstrip(",") #limpia final de lineas
                if not linea:
                    continue
                try:
                    crudos.append(json.loads(linea))
                except json.JSONDecodeError as e:
                    registros_descartados.append({"linea": i, "contenido": linea, "error": str(e)})
    else:
        crudos = ruta_o_lista  

    for registro in crudos:
        if not isinstance(registro, dict):
            registros_descartados.append({"contenido": registro, "error": "no es un objeto JSON"})
            continue

        if "sensores" in registro and isinstance(registro["sensores"], dict):
            plano = {**{k: v for k, v in registro.items() if k != "sensores"}, **registro["sensores"]}
        else:
            plano = registro
        registros_validos.append(plano) #Revisa los datos anidados 

    if registros_descartados:
        print(f"[AVISO] {len(registros_descartados)} registros con sintaxis JSON inválida fueron descartados.")

    return pd.DataFrame(registros_validos)

# Fase 1.3: Normalización de tipos numéricos 

def normalizar_tipos_numericos(df, columnas):
    for col in columnas:
        if col not in df.columns:
            continue
        if not pd.api.types.is_numeric_dtype(df[col]): 
            df[col] = (
                df[col]
                .astype(str)
                .str.replace(r"[^\d\.,\-]", "", regex=True)  # quita °C, %, espacios, texto
                .str.replace(",", ".", regex=False)           # coma decimal - punto 
            )
        df[col] = pd.to_numeric(df[col], errors="coerce")
    return df


def pipeline_limpieza_invernadero(df_crudo, config=config):
    df = df_crudo.copy()

    # 0. Timestamp como índice datetime
    df["timestamp"] = pd.to_datetime(df["timestamp"], format="mixed", dayfirst=True, errors="coerce")
    df.set_index("timestamp", inplace=True)
    df.sort_index(inplace=True)  # asfreq y diff() requieren orden cronológico, mas antiguo a mas reciente 

    # 0b. Forzar tipos numéricos (por si el JSON trajo "24.1°C" o "60,5" como texto)
    df = normalizar_tipos_numericos(df, config["columnas_numericas"])

    # 1. Eliminar registros duplicados de red
    df = df[~df.index.duplicated(keep="first")]

    # 2. Reemplazar códigos de error de hardware por NaN
    df.replace(config["error_values"], np.nan, inplace=True)

    # 2b. Ceros sospechosos en columnas donde 0 real es casi imposible
    for col in config["columnas_cero"]:
        if col in df.columns:
            df.loc[df[col] == 0, col] = np.nan

    # 3. Sincronizar el reloj (crea filas NaN en los huecos de red)
    df = df.asfreq(config["frecuencia_esperada"])

    # 4. Filtrar valores fuera de rango físico
    for col, (minimo, maximo) in config["rangos_validos"].items():
        if col in df.columns:
            df.loc[~df[col].between(minimo, maximo), col] = np.nan

    # 5. Filtrar picos imposibles (spike detection)
    for col, max_delta in config["max_delta_por_lectura"].items():
        if col in df.columns:
            df.loc[df[col].diff().abs() > max_delta, col] = np.nan

    # Fase 1.4 . RELLENO INTELIGENTE

    columnas_continuas = [c for c in config["rangos_validos"] if c != "lluvia_pct" and c in df.columns]
    promedio_movil = df[columnas_continuas].rolling(
        config["ventana_promedio_relleno"], center=True, min_periods=1
    ).mean()

    for col in columnas_continuas:
        es_nulo = df[col].isna()
        grupo_hueco = (es_nulo != es_nulo.shift()).cumsum()
        tamano_hueco = es_nulo.groupby(grupo_hueco).transform("sum")
        rellenable = es_nulo & (tamano_hueco <= config["limite_relleno"])
        df.loc[rellenable, col] = promedio_movil.loc[rellenable, col]

    # Fase 1.4.1 Lluvia
    
    if "lluvia_pct" in df.columns:
        df["lluvia_pct"] = df["lluvia_pct"].ffill(limit=config["limite_relleno_lluvia"])

    return df


if __name__ == "__main__":
    # Simula el JSON crudo tal como llegaría del nodo DHT22 (sucio)

    datos_ejemplo = [
        {"timestamp": "2026-06-18 14:00:00", "temp_ambiente": "24.1°C", "hum_ambiente": "60%", "hum_suelo": 45, "lluvia_pct": 0},
        {"timestamp": "2026-06-18 14:00:10", "temp_ambiente": 24.3, "hum_ambiente": 61, "hum_suelo": 45, "lluvia_pct": 0},
        {"timestamp": "2026-06-18 14:00:20", "temp_ambiente": 52.0, "hum_ambiente": 60, "hum_suelo": 45, "lluvia_pct": 0},  # spike
        {"timestamp": "2026-06-18 14:00:30", "temp_ambiente": 24.2, "hum_ambiente": 0,  "hum_suelo": 45, "lluvia_pct": 0},  # 0 sospechoso
        # hueco: faltan 14:00:40 y 14:00:50 (simula caída de red corta)
        {"timestamp": "2026-06-18 14:01:00", "temp_ambiente": -999, "hum_ambiente": 58, "hum_suelo": 44, "lluvia_pct": 80},  # código de error
        {"timestamp": "2026-06-18 14:01:10", "temp_ambiente": 23.9, "hum_ambiente": 58, "hum_suelo": 44, "lluvia_pct": 75},
        {"timestamp": "2026-06-18 14:01:10", "temp_ambiente": 23.9, "hum_ambiente": 58, "hum_suelo": 44, "lluvia_pct": 75},  # duplicado
    ]

    # Paso 1: carga segura (acepta lista en memoria o ruta a archivo JSON/NDJSON)
    df_crudo = cargar_json_seguro(datos_ejemplo)

    # Pasos 2-9: pipeline de limpieza completo
    resultado = pipeline_limpieza_invernadero(df_crudo)
    print(resultado.to_string())

                     temp_ambiente  hum_ambiente  hum_suelo  lluvia_pct
timestamp                                                              
2026-06-18 14:00:00           24.1          60.0  45.000000         0.0
2026-06-18 14:00:10           24.3          61.0  45.000000         0.0
2026-06-18 14:00:20           24.1          60.0  45.000000         0.0
2026-06-18 14:00:30           24.1          59.4  45.000000         0.0
2026-06-18 14:00:40           24.1          59.4  44.666667         0.0
2026-06-18 14:00:50           24.1          59.4  44.666667         0.0
2026-06-18 14:01:00           24.1          58.0  44.000000        80.0
2026-06-18 14:01:10           23.9          58.0  44.000000        75.0
